In [1]:
import torch
from transformers import  AutoProcessor,AutoModelForMultimodalLM

MODEL_ID= 'Qwen/Qwen3.5-4B'

# 1.加载processer和模型
processer=AutoProcessor.from_pretrained(MODEL_ID)
model=AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)

# 2.转为测试模式
model.eval()

/home/20261024_1/.local/share/mamba/envs/vlm-test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 723/723 [00:02<00:00, 283.38it/s]


Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

### 测试1

1. 尝试中文+非思考模式。模型能够正确回答。

In [10]:
import time

# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试1.png'
question=f'''
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 2 列的那本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
torch.cuda.synchronize()
start_time = time.perf_counter()
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

torch.cuda.synchronize()
end_time = time.perf_counter()
inference_time = end_time - start_time

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
generated_tokens = outputs[0].shape[-1] - input_length
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)
print("\n========== 推理性能 ==========")
print(f"推理耗时: {inference_time:.3f} s")
print(f"生成 Token 数: {generated_tokens}")
print(f"生成速度: {generated_tokens / inference_time:.2f} tokens/s")


Qwen3.5 回答：
根据图片内容，第 2 行、第 2 列的那本书是《Music》。

1. 这本书的书脊主要颜色是**白色**（或米白色）。
2. 书脊上的文字是：**Music**。
3. 这本书位于书架的**第 2 行、第 2 列**。


========== 推理性能 ==========
推理耗时: 2.119 s
生成 Token 数: 76
生成速度: 35.86 tokens/s


2. 尝试中文+非思考模式+直接回答指令。模型能正确回答。

In [11]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试1.png'
question=f'''
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 2 列的那本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
请直接输出答案，而不要输出你的分析过程。
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
torch.cuda.synchronize()
start_time = time.perf_counter()
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

torch.cuda.synchronize()
end_time = time.perf_counter()
inference_time = end_time - start_time

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
generated_tokens = outputs[0].shape[-1] - input_length
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)
print("\n========== 推理性能 ==========")
print(f"推理耗时: {inference_time:.3f} s")
print(f"生成 Token 数: {generated_tokens}")
print(f"生成速度: {generated_tokens / inference_time:.2f} tokens/s")

Qwen3.5 回答：
1. 这本书的书脊主要颜色是白色。
2. 书名是：Music
3. 这本书位于第 2 行、第 2 列。


========== 推理性能 ==========
推理耗时: 1.163 s
生成 Token 数: 40
生成速度: 34.40 tokens/s


### 测试5

1. 中文+非思考模式。模型能正确回答。

In [13]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试5.jpg'
question=f'''
请仔细观察图片中的书架。
我们将书架中的书籍按照从左到右的顺序排列，最左边的黄色的书作为第1本。现在请找到第14本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
torch.cuda.synchronize()
start_time = time.perf_counter()
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

torch.cuda.synchronize()
end_time = time.perf_counter()
inference_time = end_time - start_time

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
generated_tokens = outputs[0].shape[-1] - input_length
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)
print("\n========== 推理性能 ==========")
print(f"推理耗时: {inference_time:.3f} s")
print(f"生成 Token 数: {generated_tokens}")
print(f"生成速度: {generated_tokens / inference_time:.2f} tokens/s")

Qwen3.5 回答：
根据图片内容，从左到右依次排列，第14本书是《苏东坡新传》（下）。

1. 这本书的书脊主要颜色是白色。
2. 书脊上的文字是“苏东坡新传 下”。
3. 这本书位于书架的中间行，从左往右数第14本。


========== 推理性能 ==========
推理耗时: 3.685 s
生成 Token 数: 76
生成速度: 20.62 tokens/s


2. 下面尝试中文+思考模式。未翻车。

In [16]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试5.jpg'
question=f'''
请仔细观察图片中的书架。
我们将书架中的书籍按照从左到右的顺序排列，最左边的黄色的书作为第1本。现在请找到第14本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
torch.cuda.synchronize()
start_time = time.perf_counter()
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

torch.cuda.synchronize()
end_time = time.perf_counter()
inference_time = end_time - start_time

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
generated_tokens = outputs[0].shape[-1] - input_length
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)
print("\n========== 推理性能 ==========")
print(f"推理耗时: {inference_time:.3f} s")
print(f"生成 Token 数: {generated_tokens}")
print(f"生成速度: {generated_tokens / inference_time:.2f} tokens/s")

Qwen3.5 回答：
根据图片从左到右的排列，我们将书脊清晰可见、可独立计数的书籍作为计数对象。

1.  **第1本：** 最左侧黄色的《苏东坡传》。
2.  **第2本：** 浅蓝色的《数学Ⅱ(女孩)(1)》。
3.  **第3本：** 浅粉色的《数学Ⅱ(女孩)(3)》。
4.  **第4本：** 浅紫色的《数学Ⅱ(女孩)(2)》。
5.  **第5本：** 淡粉色的《数学Ⅱ(女孩)(4)》。
6.  **第6本：** 红色的《现代汉语词典》。
7.  **第7本：** 绿色的《图解电的基础知识入门》。
8.  **第8本：** 绿色的《图解电机基础知识入门》。
9.  **第9本：** 绿色的《图解机械基础知识入门》。
10. **第10本：** 白色的《电磁场基本原理66课》。
11. **第11本：** 白色的《电工电路基本原理66课》。
12. **第12本：** 白色的《电子电路基本原理66课》。
13. **第13本：** 白色与棕色结合的《苏东坡新传(上)》。
14. **第14本：** 白色与棕色结合的《苏东坡新传(下)》。

**回答：**

1.  这本书的书脊主要颜色是 **白色**。
2.  书脊上的书名是：**苏东坡新传**（书脊下部有小字标注“(下)”，表明这是分册中的第二册）。
3.  这本书位于 **最右边的一排**，在该排书籍的 **第2本** 的位置（紧邻着《苏东坡新传(上)》的右侧）。


========== 推理性能 ==========
推理耗时: 12.902 s
生成 Token 数: 415
生成速度: 32.16 tokens/s


3. 尝试英文+非思考模式。没有翻车，但是有一定的列举分析。

In [17]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试5.jpg'
question=f'''
Please carefully observe the bookshelf in the image.
We arrange the books in the bookshelf in order from left to right, with the yellow book on the far left as the 1st book. Now please find the 14th book and answer:
What is the main color of this book's spine?
If there is legible text on the spine, please accurately copy the book title.
Briefly describe the position of this book, explaining which row and which column you found it in.
Please note:
You must judge based on the actual content of the image, and do not guess based on common arrangement patterns of bookshelves.
If the text cannot be recognized, clearly answer "cannot be recognized" and do not fabricate a book title.
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

## 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
torch.cuda.synchronize()
start_time = time.perf_counter()
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

torch.cuda.synchronize()
end_time = time.perf_counter()
inference_time = end_time - start_time

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
generated_tokens = outputs[0].shape[-1] - input_length
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)
print("\n========== 推理性能 ==========")
print(f"推理耗时: {inference_time:.3f} s")
print(f"生成 Token 数: {generated_tokens}")
print(f"生成速度: {generated_tokens / inference_time:.2f} tokens/s")

Qwen3.5 回答：
Based on the image provided, here is the analysis of the bookshelf:

**14th Book Analysis:**

*   **Main Color of the Spine:** White
*   **Book Title:** 苏东坡新传 (Su Dongpo Xin Zhuan)
*   **Position Description:**
    *   **Row:** The books are arranged in a single row on the shelf.
    *   **Column:** It is the 14th book from the left.
    *   **Context:** This book is located immediately to the right of the set of three "66 Lessons" books (specifically, to the right of the "Electronic Circuit Basic Principles 66 Lessons" book). It is the first volume (marked with a small "上" for "Upper") of the "Su Dongpo New Biography" series.


========== 推理性能 ==========
推理耗时: 5.962 s
生成 Token 数: 178
生成速度: 29.86 tokens/s


4. 下面尝试英文+非思考模式+直接输入指令。翻车了。

In [18]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试5.jpg'
question=f'''
Please carefully observe the bookshelf in the image.
We arrange the books in the bookshelf in order from left to right, with the yellow book on the far left as the 1st book. Now please find the 14th book and answer:
What is the main color of this book's spine?
If there is legible text on the spine, please accurately copy the book title.
Briefly describe the position of this book, explaining which row and which column you found it in.
Please note:
You must judge based on the actual content of the image, and do not guess based on common arrangement patterns of bookshelves.
If the text cannot be recognized, clearly answer "cannot be recognized" and do not fabricate a book title.
Please output the answer directly, and do not output your analysis process.
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
torch.cuda.synchronize()
start_time = time.perf_counter()
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

torch.cuda.synchronize()
end_time = time.perf_counter()
inference_time = end_time - start_time

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
generated_tokens = outputs[0].shape[-1] - input_length
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)
print("\n========== 推理性能 ==========")
print(f"推理耗时: {inference_time:.3f} s")
print(f"生成 Token 数: {generated_tokens}")
print(f"生成速度: {generated_tokens / inference_time:.2f} tokens/s")

Qwen3.5 回答：
The 14th book is the one titled "苏东坡新传 上".
Its main spine color is white.
It is located in the second row, the 7th column from the left.


========== 推理性能 ==========
推理耗时: 2.529 s
生成 Token 数: 47
生成速度: 18.58 tokens/s
